# iTransformer + Anti-Collapse Stack + Repulsive Ensemble + DtACI

Полный SOTA-стек после R-0050 collapse-remediation. Все компоненты собраны и работают вместе:

**Архитектура**
- **iTransformer** (Liu et al., ICLR 2024, [arXiv:2310.06625](https://arxiv.org/abs/2310.06625)) — inverted attention поверх variate-токенов. Дефолт сжат до d=64, layers=2 (Sprint B4).
- **Logit Adjustment** (Menon et al., ICLR 2021, [arXiv:2007.07314](https://arxiv.org/abs/2007.07314)) — Sprint A1, активен внутри iTransformer.
- **DropPath** (Sprint B3): stochastic depth p=0.2 внутри encoder-блоков.

**Loss**
- **CompositeQuantLoss** = ASL·α + RankIC·β + Sharpe·γ + Monotone·δ (Sprint A2, D2). По умолчанию `inner_classification='asl'` и `sharpe_weight=0` — устранены trivial-минимумы на prior, которые вызвали collapse.
- **Asymmetric Loss** (Ridnik & Ben-Baruch, ICCV 2021, [arXiv:2009.14119](https://arxiv.org/abs/2009.14119)) — Sprint A2.
- **Class-Balanced pos_weight** (Cui et al., CVPR 2019) — Sprint A3.
- **Uncertainty Weighting** (Kendall CVPR 2018) — Sprint D1, опционально.

**Регуляризация**
- **ImbSAM** (Zhou et al., ICCV 2023, [arXiv:2308.07815](https://arxiv.org/abs/2308.07815)) — Sprint B1, sharpness-aware на minority-class.
- **Mixup** для time-series (Zhang ICLR 2018, Amazon Science 2024) — Sprint B2.

**Ensemble + UQ**
- **Repulsive Deep Ensemble (M=5)** (D'Angelo & Fortuin, NeurIPS 2021, [arXiv:2106.11642](https://arxiv.org/abs/2106.11642)) — Sprint C1. Sequential function-space repulsion в RBF-kernel.
- **DtACI** (Gibbs & Candès, JMLR 2024, [arXiv:2208.08401](https://arxiv.org/abs/2208.08401)) — Sprint C2. Grid γ-кандидатов с экспоненциально взвешенным агрегированием — устраняет проблему mis-specified γ из R-0050.

Acceptance-тесты после каждой фазы выводятся прямо в ноутбук — иначе collapse мог бы остаться незамеченным до бэктеста.

## 0. Окружение

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH],
                       check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard',
                        f'origin/{REPO_BRANCH}'], check=True, env=env)
    else:
        subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
             f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
            check=True, env=env,
        )
else:
    PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}, IN_COLAB: {IN_COLAB}')

In [ ]:
if IN_COLAB:
    subprocess.run(
        ['pip', 'install', '--quiet',
         'torch>=2.2', 'pandas>=2.1', 'pyarrow>=15.0',
         'scikit-learn>=1.4', 'requests>=2.31', 'tqdm>=4.66', 'ipywidgets>=8.0'],
        check=True,
    )
    print('Готово.')

In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if not (SRC / 'graduate_work' / '__init__.py').exists():
    raise FileNotFoundError(f'graduate_work не найден в {SRC}')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from graduate_work.config import default_config, ModelConfig, TradingConfig, TrainingConfig

cfg = default_config()
# Anti-collapse stack: переопределяем все ключевые поля.
cfg = dataclasses.replace(
    cfg,
    model=dataclasses.replace(
        cfg.model,
        architecture='itransformer',
        itransformer_seq_len=cfg.data.window_size,
        # Sprint B4 — уже в дефолтах, но явно фиксируем.
        itransformer_d_model=64,
        itransformer_n_layers=2,
        itransformer_drop_path=0.2,
        # Sprint A1 — Logit Adjustment включён.
        logit_adjust_tau=1.0,
    ),
    trading=dataclasses.replace(
        cfg.trading,
        # Sprint A2 + D2 — composite с ASL inner и sharpe выкл.
        loss_objective='composite',
        composite_inner_loss='asl',
        composite_sharpe_weight=0.0,
        # Sprint A3 — Class-Balanced pos_weight.
        use_class_balanced_pos_weight=True,
        # Sprint D1 — Uncertainty Weighting (опционально).
        composite_uncertainty_weighting=False,
    ),
    training=dataclasses.replace(
        cfg.training,
        # Sprint B1 — ImbSAM.
        use_imbsam=True,
        imbsam_rho=0.05,
        imbsam_horizon_index=0,
        # Sprint B2 — Mixup.
        mixup_alpha=0.2,
        mixup_p=0.5,
        # Sprint C1 — Repulsive Deep Ensemble.
        ensemble_repulsion_weight=0.1,
    ),
)
print('=== ANTI-COLLAPSE STACK CONFIGURATION ===')
print(f'arch={cfg.model.architecture}, d_model={cfg.model.itransformer_d_model}, '
      f'layers={cfg.model.itransformer_n_layers}, drop_path={cfg.model.itransformer_drop_path}')
print(f'logit_adjust_tau={cfg.model.logit_adjust_tau}')
print(f'loss_objective={cfg.trading.loss_objective}, '
      f'inner_loss={cfg.trading.composite_inner_loss}, '
      f'sharpe_w={cfg.trading.composite_sharpe_weight}')
print(f'class_balanced={cfg.trading.use_class_balanced_pos_weight}, '
      f'uncertainty_weighting={cfg.trading.composite_uncertainty_weighting}')
print(f'imbsam={cfg.training.use_imbsam}@ρ={cfg.training.imbsam_rho}, '
      f'mixup α={cfg.training.mixup_alpha}@p={cfg.training.mixup_p}')
print(f'ensemble_repulsion_weight={cfg.training.ensemble_repulsion_weight}')
print(f'tickers={cfg.data.tickers}, horizons={cfg.data.horizons}, window={cfg.data.window_size}')

## 1. Drive: данные ALGOPACK

Ожидается такая же структура, что и у `algopack_baseline.ipynb`.

In [ ]:
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    src_dir = DRIVE_BASE / 'data' / 'raw' / 'Algopack'
    if not src_dir.exists():
        src_dir = DRIVE_BASE / 'data' / 'raw'
    if src_dir.exists():
        shutil.copytree(src_dir, cfg.paths.data_raw, dirs_exist_ok=True)
        print(f'Подтянули данные из {src_dir}')
    else:
        raise FileNotFoundError(f'Не найдено {src_dir}.')

for sub in ['algopack/tradestats', 'algopack/orderstats', 'algopack/obstats']:
    p = cfg.paths.data_raw / sub
    n = sum(1 for _ in p.glob('*.csv')) if p.exists() else 0
    print(f'  {sub}: {n} файлов')

## 2. Загрузка SuperCandles

In [ ]:
def _load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def _load_supercandle(product: str, ticker: str) -> pd.DataFrame:
    p = cfg.paths.data_raw / 'algopack' / product / f'{ticker}.csv'
    df = _load_csv(p)
    if df.empty or 'tradedate' not in df.columns:
        return df
    if 'tradetime' in df.columns:
        ts = pd.to_datetime(
            df['tradedate'].astype(str) + ' ' + df['tradetime'].astype(str),
            utc=True, errors='coerce',
        )
    else:
        ts = pd.to_datetime(df['tradedate'], utc=True, errors='coerce')
    df.index = ts
    df.index.name = 'begin'
    return df.dropna(how='all').sort_index()


ts_data, os_data, ob_data = {}, {}, {}
div_data = {}
for ticker in cfg.data.tickers:
    ts_data[ticker] = _load_supercandle('tradestats', ticker)
    os_data[ticker] = _load_supercandle('orderstats', ticker)
    ob_data[ticker] = _load_supercandle('obstats', ticker)
    p = cfg.paths.data_raw / 'dividends' / f'{ticker}.csv'
    if p.exists():
        df = pd.read_csv(p)
        if 'registryclosedate' in df.columns:
            df['registryclosedate'] = pd.to_datetime(
                df['registryclosedate'], utc=True, errors='coerce',
            )
        div_data[ticker] = df

cal_dir = cfg.paths.data_raw / 'calendars'
calendars = {p.stem: pd.read_csv(p) for p in cal_dir.glob('*.csv')} if cal_dir.exists() else {}

for t, df in ts_data.items():
    print(f'  {t}: {len(df)} 5-мин баров, range {df.index.min()} .. {df.index.max() if not df.empty else "-"}')

## 3. Build feature frame (как в algopack_baseline.ipynb)

In [ ]:
from graduate_work.features.algopack_features import (
    obstats_features, orderstats_features, tradestats_features, order_to_trade_ratio,
)
from graduate_work.features.calendar_features import (
    dividend_features, trading_day_features, expirations_features,
)
from graduate_work.features.targets import cost_aware_classification_labels, lr_columns


def _ts_to_ohlcv(ts: pd.DataFrame, ticker: str) -> pd.DataFrame:
    if ts.empty:
        return pd.DataFrame()
    out = pd.DataFrame(index=ts.index)
    out['open'] = ts['pr_open'].astype(float)
    out['high'] = ts['pr_high'].astype(float)
    out['low']  = ts['pr_low'].astype(float)
    out['close']= ts['pr_close'].astype(float)
    out['volume'] = ts['vol'].astype(float)
    out['value']  = ts['val'].astype(float)
    out['vwap']   = ts['pr_vwap'].astype(float)
    out['ticker'] = ticker
    return out


def _build_log_returns(close: pd.Series) -> pd.DataFrame:
    lr = np.log(close / close.shift(1))
    return pd.DataFrame({
        'lr_1':  lr,
        'lr_5':  lr.rolling(5).sum(),
        'lr_15': lr.rolling(15).sum(),
        'lr_30': lr.rolling(30).sum(),
    }, index=close.index)


def build_ticker_frame(ticker: str) -> pd.DataFrame:
    ts_df = ts_data[ticker]
    if ts_df.empty:
        return pd.DataFrame()
    feat = _ts_to_ohlcv(ts_df, ticker)
    feat = feat.join(_build_log_returns(feat['close']), how='left')
    feat = feat.join(tradestats_features(ts_df), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(orderstats_features(os_data[ticker]), how='left')
    if not ob_data[ticker].empty:
        feat = feat.join(obstats_features(ob_data[ticker]), how='left')
    if not os_data[ticker].empty and not ts_data[ticker].empty:
        feat = feat.join(order_to_trade_ratio(os_data[ticker], ts_data[ticker]), how='left')
    trading_days = calendars.get('trading_days_stock', pd.DataFrame())
    feat = feat.join(trading_day_features(trading_days, feat.index), how='left')
    expir = calendars.get('futures_expirations', pd.DataFrame())
    feat = feat.join(expirations_features(expir, feat.index), how='left')
    if ticker in div_data and not div_data[ticker].empty:
        feat = feat.join(
            dividend_features(div_data[ticker], feat.index, last_close_price=None),
            how='left',
        )
    distance_cols = [c for c in feat.columns if c.startswith(('cal_days_', 'div_days_'))]
    if distance_cols:
        feat[distance_cols] = feat[distance_cols].ffill()
    return feat.fillna(0.0)


frames = []
for ticker in cfg.data.tickers:
    f = build_ticker_frame(ticker)
    if not f.empty:
        frames.append(f)
        print(f'  {ticker}: {f.shape}')

full = pd.concat(frames).sort_index()
print(f'\nfull frame: {full.shape}')

## 4. Targets + lr-arrays + windowing

In [ ]:
def _attach_targets_and_lr(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    out_parts = []
    for ticker, sub in frame.groupby('ticker'):
        labels = cost_aware_classification_labels(
            open_price=sub['open'], close_price=sub['close'],
            horizons=cfg.data.horizons,
            entry_cost=cfg.trading.commission_rate + cfg.trading.slippage_rate,
            exit_cost=cfg.trading.commission_rate + cfg.trading.slippage_rate,
            label_smoothing=cfg.data.label_smoothing,
            direction='long',
        )
        out_parts.append(sub.join(labels, how='left'))
    out = pd.concat(out_parts).sort_values(['ticker']).sort_index(kind='stable')
    target_cols = [f'target_h{h}' for h in cfg.data.horizons]
    lr_cols = lr_columns(cfg.data.horizons)
    return out, target_cols, lr_cols


full_with_targets, target_cols, lr_cols = _attach_targets_and_lr(full)
for c in target_cols:
    p_up = float((full_with_targets[c].dropna() >= 0.5).mean())
    print(f'  {c}: P(UP)={p_up:.3f}')

feature_cols = [c for c in full_with_targets.columns
                if c not in ('ticker',) and c not in target_cols and c not in lr_cols]
print(f'\nfeature_cols: {len(feature_cols)}')

In [ ]:
from graduate_work.features.pipeline import chronological_split, _build_arrays
from graduate_work.features.scaler import StandardScaler
from graduate_work.strategy import extract_lr_array

train_df, val_df, test_df = chronological_split(
    full_with_targets,
    cfg.data.train_ratio, cfg.data.val_ratio,
    purge_horizon=max(cfg.data.horizons),
)
test_prices_raw = test_df[['open', 'close', 'ticker']].copy()

scaler = StandardScaler()
scaler.fit(train_df, feature_cols)
train_df_s = scaler.transform(train_df)
val_df_s   = scaler.transform(val_df)
test_df_s  = scaler.transform(test_df)

train = _build_arrays(train_df_s, feature_cols, target_cols, cfg.data.window_size, desc='train')
val   = _build_arrays(val_df_s,   feature_cols, target_cols, cfg.data.window_size, desc='val')
test  = _build_arrays(test_df_s,  feature_cols, target_cols, cfg.data.window_size, desc='test')

train_lr = extract_lr_array(train_df, train, cfg.data.horizons)
val_lr   = extract_lr_array(val_df,   val,   cfg.data.horizons)
print(f'train: x={train["x"].shape}, y={train["y"].shape}, lr={train_lr.shape}')
print(f'val:   x={val["x"].shape},   y={val["y"].shape},   lr={val_lr.shape}')
print(f'test:  x={test["x"].shape},  y={test["y"].shape}')

## 5. Repulsive Deep Ensemble обучения

M=5 членов с sequential function-space repulsion. Каждый i-й член (i≥1) штрафуется за RBF-схождение с frozen-предшественниками.

Внутри каждого члена работают: ImbSAM + Mixup + DropPath + Logit-adjust + composite ASL-loss + class-balanced pos_weight.

In [ ]:
from graduate_work.model import build_model
from graduate_work.training import DeepEnsembleTrainer, ensemble_predict

ENSEMBLE_SIZE = 5
INPUT_DIM = train['x'].shape[-1]
NUM_HORIZONS = len(cfg.data.horizons)

def model_factory(seed: int):
    return build_model(input_dim=INPUT_DIM, num_horizons=NUM_HORIZONS, cfg=cfg.model)

ckpt_dir = cfg.paths.checkpoints / 'itransformer_anti_collapse'
ensemble = DeepEnsembleTrainer(
    model_factory,
    cfg.training,
    ensemble_size=ENSEMBLE_SIZE,
    data_cfg=cfg.data,
    trading_cfg=cfg.trading,
    base_seed=cfg.training.seed,
    repulsion_weight=cfg.training.ensemble_repulsion_weight,
)
history = ensemble.fit(
    train, val,
    checkpoint_dir=ckpt_dir,
    train_lr=train_lr, val_lr=val_lr,
)
print('\n=== Ensemble training summary ===')
for i, (s, h) in enumerate(zip(history.seeds, history.member_histories)):
    print(f'  member {i:02d} seed={s}: best_val_loss={h.best_val_loss:.5f} '
          f'best_epoch={h.best_epoch}')

## 6. Inference + ACCEPTANCE TESTS (P1 / P2 / P4 из R-0050)

После SOTA-стека output prob должен ШИРОКО распределяться (range > 0.15), train/val gap должен быть малым, а ensemble должен расходиться (epistemic_std > 0.02). Эти три проверки выполняются прямо в ноутбуке — провал любого означает остаточный collapse.

In [ ]:
from graduate_work.strategy import (
    AdaptiveConformalPredictor, DtACIPredictor, aci_signals_to_actions,
    ConformalSignalGenerator, SignalGenerator,
    attach_actual_targets, attach_lr_targets,
    build_predictions_frame, calibrate_bayes_threshold,
)

is_cls = cfg.data.mode == 'classification'
test_mean, test_std = ensemble_predict(
    ensemble.members, test['x'],
    batch_size=cfg.training.batch_size, apply_sigmoid=is_cls,
)
val_mean, val_std = ensemble_predict(
    ensemble.members, val['x'],
    batch_size=cfg.training.batch_size, apply_sigmoid=is_cls,
)

print('=== ACCEPTANCE TEST P1 (prediction collapse) ===')
print(f'{"horizon":>8} | {"P(UP)":>7} | {"mean":>7} | {"max":>7} | '
      f'{"range":>7} | {"epist_std":>10} | {"P1?":>4} | {"P4?":>4}')
for i, h in enumerate(cfg.data.horizons):
    p_up = float((val["y"][:, i] >= 0.5).mean())
    m = test_mean[:, i]
    s = test_std[:, i]
    range_ = float(m.max() - m.min())
    p1_pass = '✓' if range_ >= 0.15 else '✗'
    p4_pass = '✓' if s.mean() >= 0.02 else '✗'
    print(f'{h:>8} | {p_up:>7.3f} | {m.mean():>7.4f} | {m.max():>7.4f} | '
          f'{range_:>7.4f} | {s.mean():>10.4f} | {p1_pass:>4} | {p4_pass:>4}')

print('\nP2 (train/val gap):')
for i, h in enumerate(history.member_histories):
    if h.train_loss and h.val_loss:
        last_train = h.train_loss[-1]
        last_val = h.val_loss[-1]
        gap = abs(last_val - last_train)
        ratio = (last_val - last_train) / max(last_train, 1e-6)
        p2_pass = '✓' if ratio < 0.15 else '✗'
        print(f'  member {i}: train={last_train:.4f}, val={last_val:.4f}, '
              f'gap={gap:.4f}, ratio={ratio:.3f}, P2: {p2_pass}')

predictions = build_predictions_frame(
    test['timestamp'], test['ticker'], test_mean, test_std, cfg.data.horizons,
)
val_predictions = build_predictions_frame(
    val['timestamp'], val['ticker'], val_mean, val_std, cfg.data.horizons,
)

## 7. Conformal calibration: ACI vs DtACI

ACI — фиксированный γ=0.005 (как в R-0050).
DtACI — grid γ ∈ {0.001, 0.005, 0.01, 0.05} с EWA-агрегацией: автоматически переключается на лучший γ при mis-spec.

In [ ]:
val_targets = attach_actual_targets(val, cfg.data.horizons)
test_targets = attach_actual_targets(test, cfg.data.horizons)

aci = AdaptiveConformalPredictor(target_alpha=0.1, gamma=0.005)
aci.calibrate(val_predictions, val_targets)
aci_frame = aci.replay(predictions, test_targets)

dtaci = DtACIPredictor(
    target_alpha=0.1, gammas=(0.001, 0.005, 0.01, 0.05), eta=0.1,
)
dtaci.calibrate(val_predictions, val_targets)
dtaci_frame = dtaci.replay(predictions, test_targets)

print('=== ACI vs DtACI: финальное состояние ===')
print('\nACI:')
print(aci.state_summary.to_string(index=False))
print('\nDtACI:')
print(dtaci.state_summary.to_string(index=False))

print('\nACCEPTANCE TEST P5 (α-saturation):')
for h in cfg.data.horizons:
    aci_alpha = float(
        aci.state_summary[aci.state_summary['horizon'] == h]['alpha'].iloc[0]
    )
    dtaci_alpha = float(
        dtaci.state_summary[dtaci.state_summary['horizon'] == h]['alpha_combined'].iloc[0]
    )
    aci_pass = '✓' if 0.05 <= aci_alpha <= 0.45 else '✗'
    dtaci_pass = '✓' if 0.05 <= dtaci_alpha <= 0.45 else '✗'
    print(f'  h={h:>2}: ACI α={aci_alpha:.3f} ({aci_pass})  '
          f'DtACI α={dtaci_alpha:.3f} ({dtaci_pass})')

## 8. Бэктест: ACI / DtACI / split-conformal / Bayes

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest

test_prices = test_prices_raw
bars_per_year = cfg.data.bars_per_year
cost_per_trade = 2.0 * (cfg.trading.commission_rate + cfg.trading.slippage_rate)

def _bt_row(name: str, signals: pd.DataFrame, threshold_repr: str) -> dict:
    bt = run_backtest(signals, test_prices, cfg.trading)
    m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    return {
        'method': name, 'threshold': threshold_repr,
        'n_BUY': int((signals['action'] == 'BUY').sum()),
        'sharpe': m['sharpe'], 'total_return_pct': m['total_return']*100,
        'max_dd_pct': m['max_drawdown']*100, 'win_rate': m['win_rate'],
    }

rows = []
rows.append(_bt_row('ACI',
    aci_signals_to_actions(aci_frame, max_positions=cfg.trading.max_positions),
    '(adaptive)'))
rows.append(_bt_row('DtACI',
    aci_signals_to_actions(dtaci_frame, max_positions=cfg.trading.max_positions),
    '(grid γ)'))

split_gen = ConformalSignalGenerator(cfg.trading, alpha=0.1, directional=True)
split_calib = split_gen.calibrate(val_predictions, val_targets)
rows.append(_bt_row('split-conformal',
    split_gen.generate(predictions),
    f'{split_calib.threshold:.4f}'))

lr_targets_attached = attach_lr_targets(full_with_targets, val, cfg.data.horizons)
bayes_calib = calibrate_bayes_threshold(
    val_predictions, lr_targets_attached, cost_per_trade=cost_per_trade,
)
tc_bayes = dataclasses.replace(
    cfg.trading,
    probability_threshold=bayes_calib.min_expected_return,
    selection_correction='none',
)
rows.append(_bt_row('Bayes',
    SignalGenerator(tc_bayes, mode=cfg.data.mode).generate(predictions),
    f'{bayes_calib.min_expected_return:.4f}'))

summary = pd.DataFrame(rows)
print('=== Anti-Collapse Stack vs Calibration Methods ===')
print(summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# ACCEPTANCE TEST P6: положительный sharpe.
best = summary.iloc[summary['sharpe'].idxmax()]
print(f'\nЛучший метод: {best["method"]} → sharpe={best["sharpe"]:.3f} '
      f'({"✓ P6 PASS" if best["sharpe"] > 0 else "✗ P6 FAIL — collapse не устранён"})')

## 9. Диагностика: эволюция α / threshold + ensemble disagreement

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

# ACI alpha
for h, sub in aci_frame.groupby('horizon'):
    sub = sub.sort_values('timestamp')
    axes[0, 0].plot(sub['timestamp'], sub['alpha'], label=f'h={h}', linewidth=1)
axes[0, 0].axhline(0.1, ls='--', color='red', alpha=0.5)
axes[0, 0].set_title('ACI alpha (single γ=0.005)')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# DtACI alpha (combined)
for h, sub in dtaci_frame.groupby('horizon'):
    sub = sub.sort_values('timestamp')
    axes[0, 1].plot(sub['timestamp'], sub['alpha'], label=f'h={h}', linewidth=1)
axes[0, 1].axhline(0.1, ls='--', color='red', alpha=0.5)
axes[0, 1].set_title('DtACI alpha_combined (grid γ + EWA)')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

# Thresholds
for h, sub in aci_frame.groupby('horizon'):
    sub = sub.sort_values('timestamp')
    axes[1, 0].plot(sub['timestamp'], sub['threshold'], label=f'h={h}', linewidth=1)
axes[1, 0].set_title('ACI probability threshold')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

for h, sub in dtaci_frame.groupby('horizon'):
    sub = sub.sort_values('timestamp')
    axes[1, 1].plot(sub['timestamp'], sub['threshold'], label=f'h={h}', linewidth=1)
axes[1, 1].set_title('DtACI probability threshold')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Ensemble disagreement
fig, ax = plt.subplots(figsize=(11, 4))
for i, h in enumerate(cfg.data.horizons):
    ax.hist(test_std[:, i], bins=40, alpha=0.5, label=f'h={h}')
ax.set_title('Repulsive Ensemble disagreement (epistemic std) — '
             'должно быть ≥ 0.02 в среднем после Sprint C1')
ax.set_xlabel('epistemic_std'); ax.set_ylabel('count')
ax.axvline(0.02, ls='--', color='red', label='target ≥ 0.02 (P4)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Итог

Этот пайплайн объединяет 4 спринта anti-collapse remediation:

- **Sprint A** (collapse-prevention): Logit Adjustment + ASL + Class-Balanced смещают minimum loss с константного prior'а.
- **Sprint B** (regularization): ImbSAM + Mixup + DropPath + ужатый капасити снижают train/val gap.
- **Sprint C** (UQ): Repulsive Deep Ensemble + DtACI восстанавливают калибровку и эпистемическую неопределённость.
- **Sprint D** (loss balance): inner_loss=ASL + sharpe_w=0 + опциональный UW устраняют trivial-минимум composite.

Acceptance-проверки P1/P2/P4/P5/P6 в ячейках 6-8 — если хоть одна не пройдена, нужен следующий итерация анализа.